In [41]:
import os
import pandas as pd
from multiprocessing import Pool, cpu_count
from functools import partial

def process_file(filename, ground_truth_folder, algorithm_folder):
    """
    Compare a single ground truth file with the corresponding algorithm result file.
    Returns a dict with 'query' and 'match' count.
    If algorithm file is missing or empty, match is 0.
    """
    ground_truth_file = os.path.join(ground_truth_folder, filename)
    algorithm_file = os.path.join(algorithm_folder, filename)

    # Read ground truth
    try:
        ground_truth_df = pd.read_csv(ground_truth_file)
    except Exception as e:
        print(f"[ERROR] Failed to read ground truth {filename}: {e}")
        return None  # Skip if ground truth missing/corrupt

    # If algorithm file is missing → 0 matches
    if not os.path.exists(algorithm_file):
        return {'query': filename, 'match': 0}

    # Process algorithm file
    try:
        algorithm_df = pd.read_csv(algorithm_file)
        algorithm_df = algorithm_df.drop_duplicates()
        ground_truth_first_10 = ground_truth_df.head(10)

        # Compute matches
        algorithm_ids = set(algorithm_df['ID'])
        ground_truth_ids = set(ground_truth_first_10['ID'])
        match_count = len(algorithm_ids.intersection(ground_truth_ids))

        return {'query': filename, 'match': match_count}

    except Exception as e:
        print(f"[ERROR] Failed to process algorithm file {filename}: {e}")
        return {'query': filename, 'match': 0}  # Treat errors as 0 matches

def process_folder(ground_truth_folder, algorithm_folder, output_file):
    """
    Process all files in a folder in parallel using multiprocessing.
    Returns a DataFrame with 'query' and 'match' for each file.
    """
    ground_truth_files = os.listdir(ground_truth_folder)

    # Ensure output directory exists
    output_directory = os.path.dirname(output_file)
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    # Partial function for multiprocessing
    process_file_partial = partial(
        process_file,
        ground_truth_folder=ground_truth_folder,
        algorithm_folder=algorithm_folder
    )

    # Use all CPU cores
    num_processes = cpu_count()
    with Pool(processes=num_processes) as pool:
        results = pool.map(process_file_partial, ground_truth_files)

    # Filter out files where ground truth could not be read
    results = [r for r in results if r is not None]

    # Save results
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_file, index=False)
    return results_df

def main():
    ground_truth_folder = '/scratch/aa5f25/datasets/yt8m//Ground_truth/'
    algorithm_base_folder = '/scratch/aa5f25/datasets/yt8m//Results/'
    output_base_folder = '/iridisfs/scratch/aa5f25/datasets/yt8m/IntermediateResults/'

    folder_suffixes = [20, 40, 80, 100, 200, 300, 500, 800, 1000, 1200, 1500, 1700, 1900, 2100, 2300] 
    # 1700, 1900, 2100, 2300, 2500, 2700, 3000

    all_results = []
    result_strings = []

    existing_folders = os.listdir(algorithm_base_folder)

    for folder_suffix in folder_suffixes:
        folder_name = f'{folder_suffix}'
        if folder_name not in existing_folders:
            print(f"[INFO] Skipping {folder_name}, folder does not exist.")
            # Even if folder missing, consider all ground truth files with 0 matches
            ground_truth_files = os.listdir(ground_truth_folder)
            avg_zero = 0.0
            all_results.append({'folder': folder_suffix, 'average_match': avg_zero})
            result_strings.append(f"{avg_zero:.4f}")
            continue

        algorithm_folder = os.path.join(algorithm_base_folder, folder_name)
        output_file = os.path.join(output_base_folder, f'Res{folder_suffix}.csv')

        results_df = process_folder(ground_truth_folder, algorithm_folder, output_file)

        if results_df.empty:
            print(f"[INFO] No data in folder {folder_name}, treating all matches as 0.")
            avg_zero = 0.0
            all_results.append({'folder': folder_suffix, 'average_match': avg_zero})
            result_strings.append(f"{avg_zero:.4f}")
            continue

        if 'match' not in results_df.columns:
            print(f"[WARNING] 'match' column not found in results for folder {folder_name}, skipping.")
            continue

        # Compute average match over all files
        average_match = results_df['match'].mean()
        all_results.append({'folder': folder_suffix, 'average_match': average_match})
        result_strings.append(f"{average_match/10:.4f}")  # Assuming top 10 for recall fraction

        print(f"[INFO] Processed {folder_name}: Average Match = {average_match:.2f}")

    # Save summary
    summary_df = pd.DataFrame(all_results)
    summary_file = os.path.join(output_base_folder, 'SummaryResults.csv')
    summary_df.to_csv(summary_file, index=False)

    print(f"\nSummary of averages has been written to {summary_file}")
    print("Result strings:", ", ".join(result_strings))

if __name__ == '__main__':
    main()


[INFO] Processed 20: Average Match = 6.36
[INFO] Processed 40: Average Match = 7.17
[INFO] Processed 80: Average Match = 7.84
[INFO] Processed 100: Average Match = 8.02
[INFO] Processed 200: Average Match = 8.38
[INFO] Processed 300: Average Match = 8.53
[INFO] Processed 500: Average Match = 8.71
[INFO] Processed 800: Average Match = 8.84
[INFO] Processed 1000: Average Match = 8.87
[INFO] Processed 1200: Average Match = 8.91
[INFO] Processed 1500: Average Match = 8.95
[INFO] Processed 1700: Average Match = 8.96
[INFO] Processed 1900: Average Match = 8.98
[INFO] Processed 2100: Average Match = 8.99
[INFO] Processed 2300: Average Match = 9.00

Summary of averages has been written to /iridisfs/scratch/aa5f25/datasets/yt8m/IntermediateResults/SummaryResults.csv
Result strings: 0.6359, 0.7175, 0.7836, 0.8015, 0.8383, 0.8529, 0.8707, 0.8837, 0.8875, 0.8911, 0.8946, 0.8963, 0.8977, 0.8988, 0.9000


In [5]:
import sys
!{sys.executable} -m pip install matplotlib numpy

  Using cached https://files.pythonhosted.org/packages/09/03/b7b30fa81cb687d1178e085d0f01111ceaea3bf81f9330c937fb6f6c8ca0/matplotlib-3.3.4-cp36-cp36m-manylinux1_x86_64.whl
  Using cached https://files.pythonhosted.org/packages/a7/1b/cbd8ae738719b5f41592a12057ef5442e2ed5f5cb5451f8fc7e9f8875a1a/kiwisolver-1.3.1-cp36-cp36m-manylinux1_x86_64.whl
  Using cached https://files.pythonhosted.org/packages/7d/2a/2fc11b54e2742db06297f7fa7f420a0e3069fdcf0e4b57dfec33f0b08622/Pillow-8.4.0.tar.gz
  Using cached https://files.pythonhosted.org/packages/5c/f9/695d6bedebd747e5eb0fe8fad57b72fdf25411273a39791cde838d5a8f51/cycler-0.11.0-py3-none-any.whl
/home/aa5f25/.local/lib/python3.6/site-packages/setuptools/command/install.py:37: SetuptoolsDeprecationWarning: setup.py install is deprecated. Use build and pip and other standards-based tools.
  setuptools.SetuptoolsDeprecationWarning,
Exception:
Traceback (most recent call last):
  File "/usr/lib/python3.6/site-packages/pip/basecommand.py", line 215, in ma

In [4]:
!pip install matplotlib
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------
# Data
# ---------------------------

predictive_dynamic_recall = [0.7738,0.7800,0.7819,0.7828,0.7840,0.7841,0.7839,0.7834,0.7843,0.7838,0.7840]
predictive_dynamic_qps = [2292.67,1256.78,776.126,871.437,648.274,532.636,420.15,349.037,311.999,282.997,229.116]

navix_recall = [0.8536,0.8565,0.8569,0.8569,0.8569,0.8569,0.8569,0.8569,0.8569,0.8569,0.8569]
navix_qps = [1862.2,1506.02,1052.63,950.57,590.667,440.529,290.107,197.941,166.528,142.939,120.861]

acorn_recall = [0.8509,0.8551,0.8564,0.8566,0.8570,0.8570,0.8570,0.8570,0.8570,0.8570,0.8570]
acorn_qps = [1589.83,1145.48,649.351,683.995,378.358,294.898,211.193,146.563,113.559,95.5475,94.6342]

predictor_recall = [0.4202,0.5403,0.6580,0.6926,0.7753,0.8095,0.8368,0.8501,0.8530,0.8565,0.8588,0.8600,0.8603,0.8607,0.8607,0.8607,0.8607,0.8608]
predictor_qps = [7520.21,6060.9,3789.99,3541.93,2078.61,1435.68,849.131,528.203,689.229,399.274,379.3,321.638,318.375,331.588,248.323,282.83,311.993,332.515]

twohop_recall = [0.7859,0.8260,0.8475,0.8515,0.8580,0.8597,0.8610,0.8614,0.8614,0.8614,0.8614,0.8614,0.8614,0.8614,0.8614,0.8614,0.8614,0.8614]
twohop_qps = [1384.69,1561.07,1546.15,1377.37,1455.96,1369.1,1003.72,675.711,545.247,472.864,387.49,348.771,321.132,290.135,266.529,248.908,233.641,213.416]


# ---------------------------
# Function to filter recall >= 0.8
# ---------------------------

def filter_points(qps, recall, threshold=0.8):
    fqps = []
    frecall = []

    for q, r in zip(qps, recall):
        if r >= threshold:
            fqps.append(q)
            frecall.append(r)

    return fqps, frecall


# Apply filtering
pd_qps, pd_rec = filter_points(predictive_dynamic_qps, predictive_dynamic_recall)
nav_qps, nav_rec = filter_points(navix_qps, navix_recall)
ac_qps, ac_rec = filter_points(acorn_qps, acorn_recall)
pr_qps, pr_rec = filter_points(predictor_qps, predictor_recall)
th_qps, th_rec = filter_points(twohop_qps, twohop_recall)


# ---------------------------
# Plot
# ---------------------------

plt.figure(figsize=(6,4))

plt.plot(pd_qps, pd_rec, marker='o', label="Predictive Dynamic Query")
plt.plot(nav_qps, nav_rec, marker='o', label="NAVIX")
plt.plot(ac_qps, ac_rec, marker='o', label="ACORN-1")
plt.plot(pr_qps, pr_rec, marker='o', label="Proposed Predictor")
plt.plot(th_qps, th_rec, marker='o', label="Post Filter Two-Hop")

plt.xscale("log")

plt.xlabel("QPS (Log Scale)", fontsize=14)
plt.ylabel("Recall (>=0.8)", fontsize=14)

plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

/bin/bash: pip: command not found


ModuleNotFoundError: No module named 'matplotlib'

In [2]:
import pandas as pd

df_queries = pd.read_csv('/scratch/aa5f25/datasets/yt8m/Queries_for_conjunctive_disjunctive.csv', sep=";")
df_queries

,id,video,audio,start_range,end_range,genre
0,deSR,"0.0616164207,-1.00776923,0.726923585,1.4439954...","0.770636,-0.360448927,-1.39757311,0.121328838,...",1349,2608,Nonprofits & Activism
1,9E9h,"0.410531461,0.747263491,-0.825599253,-0.709939...","-0.0263835788,-1.0880307,0.777590275,-0.682958...",4464,38450,Howto & Style
2,P6Cq,"-0.111586325,0.744629443,-0.799393654,-0.00393...","-0.454693735,-1.1063509,0.378908336,-1.0537979...",11310,46750,Autos & Vehicles
3,YYX3,"0.013093546,0.270139307,-0.858854175,0.1951588...","-0.660370529,-0.67108947,-0.120762661,0.868989...",11407,32454,Howto & Style
4,qeE7,"-0.484161347,0.386270016,1.73121119,-0.1305927...","1.2543484,-0.031926062,0.421093553,-0.20060579...",20879,20977,Music
...,...,...,...,...,...,...
1495,NHsY,"-0.0219669119,0.0346507356,-0.343290448,1.5682...","-1.65493262,1.7869792,-0.953952193,1.98489583,...",1288,2336,Sports
1496,hffa,"-1.93749464,-0.483324766,-1.37655354,0.4070281...","0.0180608667,-1.12264502,-0.393180966,-0.07987...",7821,28455,Gaming
1497,wDfq,"0.690078318,0.77443558,-0.22730732,-0.05580405...","-0.106087282,-0.155150458,-0.361773551,0.80441...",3584,4426,People & Blogs
1498,CxCu,"0.362154245,-1.2555629,-1.17699146,1.42959118,...","1.24919903,-0.553111851,-0.9114452,-0.00920430...",9041,56063,Science & Technology


In [5]:
import pandas as pd

df_data = pd.read_csv('/scratch/aa5f25/datasets/yt8m/yt_data.csv', sep=";")

df_data

,id,video,audio,genre,publication_date,views,likes
0,QSxm,"0.124597676,0.90365392,-0.554824889,0.80272740...","-0.0193368215,-0.166460291,-0.00787377451,-1.0...",Sports,2010-07-15T05:12:56-07:00,28691,31
1,aZdt,"-1.41373,0.343185037,0.450740606,-0.0968149528...","0.544073939,-0.354174435,0.696021676,-0.985756...",Music,2008-01-27T21:12:47-08:00,17490,48
2,aMAh,"-0.772988141,0.603400707,1.04376018,0.00372753...","-0.611386836,1.59351516,-0.640145,-0.989818215...",People & Blogs,2011-08-15T04:42:38-07:00,1082,2
3,zPmq,"-0.0472207256,-0.848008335,-0.438786894,0.8567...","-0.575828493,-1.33493912,0.223252654,0.2264030...",People & Blogs,2011-12-15T06:19:54-08:00,7120,15
4,FRGI,"-0.87061888,-0.718984902,-0.185527056,-0.65300...","0.270993322,-0.601711333,-0.285744935,0.190445...",Film & Animation,2009-08-17T11:01:31-07:00,27395,221
...,...,...,...,...,...,...,...
1289614,nzQg,"-0.632879,1.21565568,-0.194253698,-0.612723053...","0.801065743,-0.730201423,0.361006677,-0.847932...",Music,2013-09-01T12:52:07-07:00,22864,0
1289615,mAfi,"0.647341907,1.0648582,-0.899115622,-0.03935089...","1.1621654,-0.398566574,-0.391403198,0.27840074...",People & Blogs,2011-09-06T13:15:10-07:00,5187,7
1289616,DTH1,"-1.08109784,-0.383572906,-0.576565504,0.417198...","0.358182162,1.22967041,1.51228046,-0.389358819...",Music,2011-12-19T22:50:17-08:00,3759,11
1289617,erlq,"-0.831704855,-1.95618951,-1.0715239,-0.0882156...","-1.30953288,0.118722506,0.846927643,1.26150787...",News & Politics,2015-03-13T22:32:18-07:00,2029,12


In [4]:
import pandas as pd

# Example setup (replace with your actual DataFrames)
# df_queries has 1500 rows
# df_doc has 1,289,619 rows
# Both have a 'genre' column

# Step 1: Total number of documents
total_docs = len(df_data)

# Step 2: Compute selectivity per query
selectivities = []

for genre_value in df_queries['genre']:
    # Count how many docs match this query's genre
    count = (df_data['genre'] == genre_value).sum()
    
    # Compute selectivity
    selectivity = count / total_docs
    selectivities.append(selectivity)

# Step 3: Average selectivity over all queries
average_selectivity = sum(selectivities) / len(selectivities)

print("Average selectivity:", average_selectivity)

Average selectivity: 0.1102397467779244


In [6]:
import pandas as pd

# Example setup
# df_queries has columns: 'start_range', 'end_range' (e.g., 1500 rows)
# df_data has column: 'Value' (e.g., 1,289,619 rows)

# Step 1: Total number of documents
total_docs = len(df_data)

# Step 2: Compute selectivity per query (range query)
selectivities = []

for _, query in df_queries.iterrows():
    start = query['start_range']
    end = query['end_range']
    
    # Count how many docs have Value in the range [start, end]
    count = df_data[(df_data['views'] >= start) & (df_data['views'] <= end)].shape[0]
    
    # Compute selectivity
    selectivity = count / total_docs
    selectivities.append(selectivity)

# Step 3: Average selectivity over all queries
average_selectivity = sum(selectivities) / len(selectivities)

print("Average selectivity:", average_selectivity)

Average selectivity: 0.33669046697771415


In [12]:
# Save df1 to a CSV file named 'f.csv'
df.to_csv('/scratch/aa5f25/datasets/yt8m/Queries_for_conjunctive_disjunctive.csv', index=False, sep=";")

In [21]:
# Install packages locally for your user
!python3 -m pip install --user --upgrade numpy pandas scikit-learn statsmodels packaging

Requirement already up-to-date: numpy in /iridisfs/home/aa5f25/.local/lib/python3.6/site-packages
Requirement already up-to-date: pandas in /iridisfs/home/aa5f25/.local/lib/python3.6/site-packages
Requirement already up-to-date: scikit-learn in /iridisfs/home/aa5f25/.local/lib/python3.6/site-packages
Requirement already up-to-date: statsmodels in /iridisfs/home/aa5f25/.local/lib/python3.6/site-packages
  Cache entry deserialization failed, entry ignored
    100% |████████████████████████████████| 40kB 6.1MB/s eta 0:00:01
Requirement already up-to-date: pytz>=2017.2 in /iridisfs/home/aa5f25/.local/lib/python3.6/site-packages (from pandas)
Requirement already up-to-date: python-dateutil>=2.7.3 in /iridisfs/home/aa5f25/.local/lib/python3.6/site-packages (from pandas)
Requirement already up-to-date: scipy>=0.19.1 in /iridisfs/home/aa5f25/.local/lib/python3.6/site-packages (from scikit-learn)
Requirement already up-to-date: joblib>=0.11 in /iridisfs/home/aa5f25/.local/lib/python3.6/site-pac

In [25]:
import numpy as np
import time

# 1. Setup Synthetic CDF Data (S-curve)
np.random.seed(42)
n = 1000000
x = np.sort(np.random.uniform(0, 100, n))
y = 1 / (1 + np.exp(-(x - 50) / 10))  # Standard Sigmoid CDF

# --- MODEL 1: SIMPLE LINEAR REGRESSION (Your C++ Logic) ---
def train_slr(x, y):
    n_pts = len(x)
    sum_x, sum_y = np.sum(x), np.sum(y)
    sum_xx, sum_xy = np.sum(x*x), np.sum(x*y)
    denominator = (n_pts * sum_xx - sum_x**2)
    slope = (n_pts * sum_xy - sum_x * sum_y) / denominator
    intercept = (sum_y - slope * sum_x) / n_pts
    return slope, intercept

# --- MODEL 2: PIECEWISE LINEAR (3-Segment) ---
def train_pw(x, y, k=3):
    segments = []
    x_splits = np.array_split(x, k)
    y_splits = np.array_split(y, k)
    for i in range(k):
        segments.append(train_slr(x_splits[i], y_splits[i]))
    return segments

# --- MODEL 3: QUANTILE REGRESSION (Simplified Median Fit) ---
# Quantile regression usually requires Linear Programming. 
# Here we use an iterative 'Iteratively Reweighted Least Squares' approximation.
def train_quant(x, y, iterations=10):
    m, b = train_slr(x, y) # Start with SLR estimate
    for _ in range(iterations):
        resid = np.abs(y - (m * x + b))
        weights = 1.0 / np.maximum(resid, 1e-6) # Weights for MAD
        sum_w = np.sum(weights)
        m = (sum_w * np.sum(weights*x*y) - np.sum(weights*x) * np.sum(weights*y)) / \
            (sum_w * np.sum(weights*x**2) - (np.sum(weights*x)**2))
        b = (np.sum(weights*y) - m * np.sum(weights*x)) / sum_w
    return m, b

# --- EVALUATION ENGINE ---
def benchmark(name, train_fn, predict_fn):
    # Training
    t0 = time.time()
    model_params = train_fn()
    t_train = time.time() - t0
    
    # Inference (Average over n samples)
    t1 = time.time()
    preds = predict_fn(model_params)
    t_inf = (time.time() - t1) / n
    
    # Accuracy
    mae = np.mean(np.abs(y - preds))
    print(f"{name:<20} | {t_train:.6f}s | {t_inf:.8f}s | MAE: {mae:.6f}")

# Run the tests
print(f"{'Model Type':<20} | {'Train Time':<10} | {'Inf/Sample':<10} | Error (MAE)")
print("-" * 75)

benchmark("Simple Linear", lambda: train_slr(x, y), lambda p: p[0]*x + p[1])

def pred_pw(p):
    res = np.zeros(n)
    chunks = np.array_split(np.arange(n), 3)
    for i, (m, b) in enumerate(p):
        res[chunks[i]] = m * x[chunks[i]] + b
    return res
benchmark("Piecewise (3-seg)", lambda: train_pw(x, y), pred_pw)

benchmark("Quantile (Median)", lambda: train_quant(x, y), lambda p: p[0]*x + p[1])

Model Type           | Train Time | Inf/Sample | Error (MAE)
---------------------------------------------------------------------------
Simple Linear        | 0.001518s | 0.00000000s | MAE: 0.078756
Piecewise (3-seg)    | 0.002058s | 0.00000000s | MAE: 0.011450
Quantile (Median)    | 0.076338s | 0.00000000s | MAE: 0.078662


In [38]:
import numpy as np
import time

# ------------------------------
# 1. Setup Synthetic CDF Data (S-curve)
# ------------------------------
np.random.seed(42)
n = 1_000_000
x = np.sort(np.random.uniform(0, 100, n))
y = 1 / (1 + np.exp(-(x - 50) / 10))  # Standard Sigmoid CDF

# ------------------------------
# 2. Simple Linear Regression (SLR)
# ------------------------------
def train_slr(x, y):
    n_pts = len(x)
    sum_x, sum_y = np.sum(x), np.sum(y)
    sum_xx, sum_xy = np.sum(x*x), np.sum(x*y)
    denominator = (n_pts * sum_xx - sum_x**2)
    slope = (n_pts * sum_xy - sum_x * sum_y) / denominator
    intercept = (sum_y - slope * sum_x) / n_pts
    return slope, intercept

# ------------------------------
# 3. Piecewise Linear Regression (3 Segments)
# ------------------------------
def train_pw(x, y, k=3):
    segments = []
    x_splits = np.array_split(x, k)
    y_splits = np.array_split(y, k)
    for i in range(k):
        segments.append(train_slr(x_splits[i], y_splits[i]))
    return segments

def pred_pw(p):
    res = np.zeros(n)
    chunks = np.array_split(np.arange(n), 3)
    for i, (m, b) in enumerate(p):
        res[chunks[i]] = m * x[chunks[i]] + b
    return res

# ------------------------------
# 4. Quantile Regression (Median) Approximation
# ------------------------------
def train_quant(x, y, iterations=10):
    m, b = train_slr(x, y)  # Start with SLR estimate
    for _ in range(iterations):
        resid = np.abs(y - (m*x + b))
        weights = 1.0 / np.maximum(resid, 1e-6)  # Avoid division by zero
        sum_w = np.sum(weights)
        m = (sum_w * np.sum(weights*x*y) - np.sum(weights*x) * np.sum(weights*y)) / \
            (sum_w * np.sum(weights*x**2) - (np.sum(weights*x)**2))
        b = (np.sum(weights*y) - m * np.sum(weights*x)) / sum_w
    return m, b

# ------------------------------
# 5. Benchmarking Function
# ------------------------------
def benchmark(name, train_fn, predict_fn, batch_size=1000):
    # Training
    t0 = time.time()
    model_params = train_fn()
    t_train = time.time() - t0

    # Inference (measure in batches for meaningful per-sample time)
    t1 = time.time()
    preds = np.zeros(n)
    for i in range(0, n, batch_size):
        end = min(i+batch_size, n)
        preds[i:end] = predict_fn(model_params)[i:end]
    t_inf_total = time.time() - t1
    t_inf_per_sample = t_inf_total / n

    # Accuracy (MAE)
    mae = np.mean(np.abs(y - preds))
    print(f"{name:<20} | {t_train:.6f}s | {t_inf_per_sample:.8f}s | MAE: {mae:.6f}")

# ------------------------------
# 6. Run Benchmarks
# ------------------------------
print(f"{'Model Type':<20} | {'Train Time':<10} | {'Inf/Sample':<12} | Error (MAE)")
print("-"*75)

# Simple Linear Regression
benchmark("Simple Linear", lambda: train_slr(x, y), lambda p: p[0]*x + p[1])

# Piecewise Linear Regression (3 Segments)
benchmark("Piecewise (3-seg)", lambda: train_pw(x, y), pred_pw)

# Quantile Regression (Median)
benchmark("Quantile (Median)", lambda: train_quant(x, y), lambda p: p[0]*x + p[1])

Model Type           | Train Time | Inf/Sample   | Error (MAE)
---------------------------------------------------------------------------
Simple Linear        | 0.001674s | 0.00000031s | MAE: 0.078756
Piecewise (3-seg)    | 0.001692s | 0.00000279s | MAE: 0.011450
Quantile (Median)    | 0.078418s | 0.00000031s | MAE: 0.078662


In [37]:
import numpy as np
import time

# ------------------------------
# 1. Setup Synthetic CDF Data (Linear Clamped)
# ------------------------------
np.random.seed(42)
n = 1_000_000
x = np.sort(np.random.uniform(0, 100, n))

# NEW: We create a linear trend and "clamp" it to [0, 1]
# This mimics your C++: std::max(0.0, std::min(1.0, val))
y_raw = 0.015 * x - 0.25 
y = np.clip(y_raw, 0, 1)  # This is the "True" CDF your code expects

# ------------------------------
# 2. Adjusted Simple Linear Regression (SLR)
# ------------------------------
def train_slr(x, y):
    n_pts = len(x)
    sum_x, sum_y = np.sum(x), np.sum(y)
    sum_xx, sum_xy = np.sum(x*x), np.sum(x*y)
    
    denominator = (n_pts * sum_xx - sum_x**2)
    if denominator == 0: return 0, 0
    
    slope = (n_pts * sum_xy - sum_x * sum_y) / denominator
    intercept = (sum_y - slope * sum_x) / n_pts
    return slope, intercept

# This function matches your C++: return std::max(0.0, std::min(1.0, predicted_value));
def predict_clamped(x, params):
    slope, intercept = params
    return np.clip(slope * x + intercept, 0, 1)

# ------------------------------
# 3. Execution
# ------------------------------
t0 = time.time()
params = train_slr(x, y)
t_train = time.time() - t0

preds = predict_clamped(x, params)
mae = np.mean(np.abs(y - preds))

print(f"{'Model':<20} | {'Train Time':<10} | {'Error (MAE)'}")
print("-" * 50)
print(f"{'Simple Linear (Adj)':<20} | {t_train:.6f}s | {mae:.6f}")

Model                | Train Time | Error (MAE)
--------------------------------------------------
Simple Linear (Adj)  | 0.001423s | 0.028991


In [34]:
import numpy as np
import time


# -------------------------------
# 1. Generate Nonlinear CDF Data
# -------------------------------
np.random.seed(42)
n = 100000
x = np.sort(np.random.uniform(0, 100, n))

# Nonlinear CDF (e.g., combination of sigmoid + sine for nonlinearity)
y = 1 / (1 + np.exp(-(x - 50) / 8)) + 0.05 * np.sin(0.2 * x)
y = np.clip(y, 0, 1)  # Keep within [0, 1]

# -------------------------------
# 2. Simple Linear Regression
# -------------------------------
def train_slr(x, y):
    n_pts = len(x)
    sum_x, sum_y = np.sum(x), np.sum(y)
    sum_xx, sum_xy = np.sum(x*x), np.sum(x*y)
    denom = n_pts * sum_xx - sum_x**2
    slope = (n_pts * sum_xy - sum_x * sum_y) / denom
    intercept = (sum_y - slope * sum_x) / n_pts
    return slope, intercept

# -------------------------------
# 3. Piecewise Linear Regression
# -------------------------------
def train_pw(x, y, k=3):
    segments = []
    x_splits = np.array_split(x, k)
    y_splits = np.array_split(y, k)
    for xi, yi in zip(x_splits, y_splits):
        segments.append(train_slr(xi, yi))
    return segments

def predict_pw(segments, x, k=3):
    res = np.zeros_like(x)
    chunks = np.array_split(np.arange(len(x)), k)
    for i, (m, b) in enumerate(segments):
        res[chunks[i]] = m * x[chunks[i]] + b
    return res

# -------------------------------
# 4. Quantile Regression (Median)
# -------------------------------
def train_quant(x, y, iterations=15):
    # Start with simple linear regression
    m, b = train_slr(x, y)
    for _ in range(iterations):
        resid = y - (m * x + b)
        weights = 1.0 / np.maximum(np.abs(resid), 1e-6)  # inverse absolute residuals
        W = weights
        sum_W = np.sum(W)
        m = (sum_W * np.sum(W*x*y) - np.sum(W*x) * np.sum(W*y)) / \
            (sum_W * np.sum(W*x*x) - (np.sum(W*x)**2))
        b = (np.sum(W*y) - m * np.sum(W*x)) / sum_W
    return m, b

# -------------------------------
# 5. Benchmark Function
# -------------------------------
def benchmark(name, train_fn, predict_fn):
    # Training
    t0 = time.time()
    model_params = train_fn()
    t_train = time.time() - t0
    
    # Inference (per sample)
    t1 = time.time()
    preds = predict_fn(model_params)
    t_inf = (time.time() - t1) / len(x)
    
    # Accuracy
    mae = np.mean(np.abs(y - preds))
    
    print(f"{name:<20} | Train: {t_train:.4f}s | Inf/Sample: {t_inf:.8f}s | MAE: {mae:.6f}")
    return preds

# -------------------------------
# 6. Run Benchmarks
# -------------------------------
print(f"{'Model Type':<20} | Train Time | Inf/Sample | MAE")
print("-"*75)

# Simple Linear
pred_slr = benchmark("Simple Linear", lambda: train_slr(x, y), lambda p: p[0]*x + p[1])

# Piecewise Linear
pred_pw_vals = benchmark("Piecewise (3-seg)", lambda: train_pw(x, y), lambda p: predict_pw(p, x))

# Quantile Regression
pred_quant = benchmark("Quantile (Median)", lambda: train_quant(x, y), lambda p: p[0]*x + p[1])

# -------------------------------
# 7. Plot the results
# -------------------------------


Model Type           | Train Time | Inf/Sample | MAE
---------------------------------------------------------------------------
Simple Linear        | Train: 0.0004s | Inf/Sample: 0.00000000s | MAE: 0.106999
Piecewise (3-seg)    | Train: 0.0006s | Inf/Sample: 0.00000001s | MAE: 0.020594
Quantile (Median)    | Train: 0.0104s | Inf/Sample: 0.00000000s | MAE: 0.105499


In [39]:
import numpy as np
import time

# ------------------------------
# 1. Setup Synthetic CDF Data (Clamped Linear)
# ------------------------------
np.random.seed(42)
n = 1_000_000
x = np.sort(np.random.uniform(0, 100, n))

# This matches your C++ logic: a linear path that hits 0 and 1
y_raw = 0.015 * x - 0.25 
y = np.clip(y_raw, 0, 1) 

# ------------------------------
# 2. Simple Linear Regression (Your C++ Logic)
# ------------------------------
def train_slr(x_data, y_data):
    n_pts = len(x_data)
    sum_x, sum_y = np.sum(x_data), np.sum(y_data)
    sum_xx, sum_xy = np.sum(x_data*x_data), np.sum(x_data*y_data)
    denom = (n_pts * sum_xx - sum_x**2)
    if denom == 0: return 0, 0
    m = (n_pts * sum_xy - sum_x * sum_y) / denom
    b = (sum_y - m * sum_x) / n_pts
    return m, b

# ------------------------------
# 3. Piecewise Linear (Flexibility Test)
# ------------------------------
def train_pw(x_data, y_data, k=3):
    return [train_slr(xs, ys) for xs, ys in zip(np.array_split(x_data, k), np.array_split(y_data, k))]

def pred_pw(params, x_data):
    res = np.zeros(len(x_data))
    chunks = np.array_split(np.arange(len(x_data)), len(params))
    for i, (m, b) in enumerate(params):
        res[chunks[i]] = np.clip(m * x_data[chunks[i]] + b, 0, 1)
    return res

# ------------------------------
# 4. Quantile Regression (Corrected IRLS)
# ------------------------------
def train_quant(x_data, y_data, iterations=15):
    # Start with a standard linear fit
    m, b = train_slr(x_data, y_data)
    for _ in range(iterations):
        preds = m * x_data + b
        diff = np.abs(y_data - preds)
        # Weighting for LAD (Least Absolute Deviations / Median)
        # We add a small epsilon to avoid division by zero
        weights = 1.0 / np.maximum(diff, 1e-4) 
        
        sw = np.sum(weights)
        swx = np.sum(weights * x_data)
        swy = np.sum(weights * y_data)
        swxx = np.sum(weights * x_data**2)
        swxy = np.sum(weights * x_data * y_data)
        
        denom = (sw * swxx - swx**2)
        if denom == 0: break
        m = (sw * swxy - swx * swy) / denom
        b = (swy - m * swx) / sw
    return m, b

# ------------------------------
# 5. Run Benchmarks
# ------------------------------
print(f"{'Model Type':<20} | {'Train Time':<10} | {'MAE'}")
print("-" * 50)

# 1. Simple Linear (The Baseline)
t0 = time.time()
p_slr = train_slr(x, y)
t_slr = time.time() - t0
mae_slr = np.mean(np.abs(y - np.clip(p_slr[0]*x + p_slr[1], 0, 1)))
print(f"{'Simple Linear':<20} | {t_slr:.6f}s | {mae_slr:.6f}")

# 2. Piecewise (The Precision Suggestion)
t0 = time.time()
p_pw = train_pw(x, y)
t_pw = time.time() - t0
mae_pw = np.mean(np.abs(y - pred_pw(p_pw, x)))
print(f"{'Piecewise (3-seg)':<20} | {t_pw:.6f}s | {mae_pw:.6f}")

# 3. Quantile (The Robustness Suggestion)
t0 = time.time()
p_q = train_quant(x, y)
t_q = time.time() - t0
mae_q = np.mean(np.abs(y - np.clip(p_q[0]*x + p_q[1], 0, 1)))
print(f"{'Quantile (Median)':<20} | {t_q:.6f}s | {mae_q:.6f}")

Model Type           | Train Time | MAE
--------------------------------------------------
Simple Linear        | 0.001775s | 0.028991
Piecewise (3-seg)    | 0.001665s | 0.015636
Quantile (Median)    | 0.090620s | 0.014046


In [40]:
import numpy as np
import time

# ------------------------------
# 1. Setup Synthetic CDF Data
# ------------------------------
np.random.seed(42)
n = 1_000_000
x = np.sort(np.random.uniform(0, 100, n))
y_raw = 0.015 * x - 0.25 
y = np.clip(y_raw, 0, 1) 

# ------------------------------
# 2. Model Functions
# ------------------------------
def train_slr(x_data, y_data):
    n_pts = len(x_data)
    sum_x, sum_y = np.sum(x_data), np.sum(y_data)
    sum_xx, sum_xy = np.sum(x_data*x_data), np.sum(x_data*y_data)
    denom = (n_pts * sum_xx - sum_x**2)
    if denom == 0: return 0, 0
    m = (n_pts * sum_xy - sum_x * sum_y) / denom
    b = (sum_y - m * sum_x) / n_pts
    return m, b

def train_pw(x_data, y_data, k=3):
    return [train_slr(xs, ys) for xs, ys in zip(np.array_split(x_data, k), np.array_split(y_data, k))]

def train_quant(x_data, y_data, iterations=10):
    m, b = train_slr(x_data, y_data)
    for _ in range(iterations):
        preds = m * x_data + b
        weights = 1.0 / np.maximum(np.abs(y_data - preds), 1e-4) 
        sw, swx, swy, swxx, swxy = np.sum(weights), np.sum(weights*x_data), np.sum(weights*y_data), np.sum(weights*x_data**2), np.sum(weights*x_data*y_data)
        denom = (sw * swxx - swx**2)
        if denom == 0: break
        m, b = (sw * swxy - swx * swy) / denom, (swy - m * swx) / sw
    return m, b

# ------------------------------
# 3. Benchmarking Inference
# ------------------------------
def run_benchmark():
    print(f"{'Model Type':<20} | {'Train (s)':<10} | {'Inf/Sample (ns)':<15} | {'MAE'}")
    print("-" * 65)

    # --- Simple Linear ---
    t0 = time.time()
    p_slr = train_slr(x, y)
    t_train = time.time() - t0
    
    t1 = time.time()
    # Logic matching your C++ predict_int()
    preds_slr = np.clip(p_slr[0] * x + p_slr[1], 0, 1)
    t_inf = (time.time() - t1) / n * 1e9 # Convert to nanoseconds
    mae = np.mean(np.abs(y - preds_slr))
    print(f"{'Simple Linear':<20} | {t_train:.6f} | {t_inf:.2f} ns        | {mae:.6f}")

    # --- Piecewise (3-seg) ---
    t0 = time.time()
    p_pw = train_pw(x, y)
    t_train = time.time() - t0
    
    t1 = time.time()
    # In C++, Piecewise requires finding the segment (branching/logic)
    res_pw = np.zeros(n)
    chunks = np.array_split(np.arange(n), 3)
    for i, (m, b) in enumerate(p_pw):
        res_pw[chunks[i]] = np.clip(m * x[chunks[i]] + b, 0, 1)
    t_inf = (time.time() - t1) / n * 1e9
    mae = np.mean(np.abs(y - res_pw))
    print(f"{'Piecewise (3-seg)':<20} | {t_train:.6f} | {t_inf:.2f} ns        | {mae:.6f}")

    # --- Quantile ---
    t0 = time.time()
    p_q = train_quant(x, y)
    t_train = time.time() - t0
    
    t1 = time.time()
    preds_q = np.clip(p_q[0] * x + p_q[1], 0, 1)
    t_inf = (time.time() - t1) / n * 1e9
    mae = np.mean(np.abs(y - preds_q))
    print(f"{'Quantile (Median)':<20} | {t_train:.6f} | {t_inf:.2f} ns        | {mae:.6f}")

run_benchmark()

Model Type           | Train (s)  | Inf/Sample (ns) | MAE
-----------------------------------------------------------------
Simple Linear        | 0.001657 | 0.94 ns        | 0.028991
Piecewise (3-seg)    | 0.001681 | 3.59 ns        | 0.015636
Quantile (Median)    | 0.060282 | 1.02 ns        | 0.021077


In [ ]:
import pandas as pd

df = pd.read_csv('/scratch/aa5f25/datasets/sift/sift_docs.csv', sep=",")
df

,value,actual_cdf,predicted_cdf,difference
0,0,0.000141,0.000000,0.000141
1,16,0.000162,0.000000,0.000162
2,30,0.000182,0.000000,0.000182
3,38,0.000202,0.000000,0.000202
4,57,0.000222,0.000000,0.000222
...,...,...,...,...
22327,65492,0.999919,0.999994,0.000075
22328,65502,0.999939,1.000000,0.000061
22329,65507,0.999960,1.000000,0.000040
22330,65517,0.999980,1.000000,0.000020


In [63]:
avg_diff = df["difference"].mean()
print(avg_diff)

0.0011932167253742207


In [4]:
import os
import pandas as pd

folder_path = "/iridisfs/scratch/aa5f25/datasets/RegressionModelTestingPrediction"

file_maes = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)

        # assume column contains errors
        # change "difference" if your column name is different
        if "difference" in df.columns:
            values = df["difference"]
        else:
            values = df.iloc[:, 0]

        mae = values.abs().mean()
        file_maes.append(mae)

# final average across files
final_average_mae = sum(file_maes) / len(file_maes) if file_maes else 0

print("Average MAE across files:", final_average_mae)

Average MAE across files: 0.10626458463380258
